# Práctica PLN mediante Deep Learning 26/27 — Multi-Author Writing Style Analysis (PAN 2026)

**Integrantes**
- Sergio del Barco
- Jorge Comesaña

## Tabla resumen de experimentos

> F1-macro sobre el conjunto de **test** (métrica oficial de PAN, calculada por separado para cada dificultad). La última celda del notebook imprime esta tabla con los valores reales para copiarla aquí.

| # | Experimento | Entrenamiento | F1 easy | F1 medium | F1 hard | F1 medio | Conclusión |
|---|---|---|---|---|---|---|---|
| 0 | Baseline: siempre «sin cambio» | — | _pend._ | _pend._ | _pend._ | _pend._ | Cota inferior: F1-macro ≈ 0,5 · F1 de la clase 0 |
| 1 | Baseline: aleatorio estratificado | Tasa de cambios de train | _pend._ | _pend._ | _pend._ | _pend._ | _pend._ |
| 2 | RoBERTa-base, pares de oraciones, umbral 0,5 | 40k pares/dificultad | _pend._ | _pend._ | _pend._ | _pend._ | _pend._ |
| 3 | RoBERTa-base + umbral global ajustado en validación | Ídem | _pend._ | _pend._ | _pend._ | _pend._ | _pend._ |
| 4 | RoBERTa-base + umbral por dificultad ajustado en validación | Ídem | _pend._ | _pend._ | _pend._ | _pend._ | _pend._ |

## Metodología experimental

1. **Tarea**: el problema se plantea como una **clasificación binaria de pares de oraciones consecutivas**, en la que hay que decidir si hay cambio de autor (`1`) o no (`0`) entre la oración *i* y la *i+1*.
2. **Métrica**: F1-macro sobre todos los pares de cada dificultad (*easy*, *medium* y *hard*), igual que el evaluador oficial de PAN. También se da la media de las tres.
3. **Particiones**:
   - `train` se usa para entrenar, con un submuestreo de pares por limitación de hardware.
   - `validation` sirve para elegir el checkpoint (con un subconjunto) y para ajustar el umbral de decisión (con el conjunto completo).
   - `test` **solo** se usa para la evaluación final.
4. **Reproducibilidad**: semilla fija (`SEED = 42`) y todos los hiperparámetros en una única celda de configuración.
5. **Hardware**: GPU RTX 3050 Laptop de 4 GB. Se aplican las técnicas del cuaderno de optimización (precisión mixta bf16, TF32, atención SDPA, AdamW fusionado, *padding* dinámico con agrupación por longitud, acumulación de gradiente y elección del tamaño de batch midiendo memoria). También se miden el tiempo y la memoria de cada fase.
6. **Comparación**: los modelos se comparan contra los baselines triviales de este notebook y contra los resultados publicados de PAN para la misma tarea.

## 0. Descripción de la tarea y de los datos

**Multi-Author Writing Style Analysis 2026**: dado un documento escrito por uno o varios autores, hay que detectar **en qué posiciones cambia el autor**, a nivel de oración.

El dataset `mawsa_full_dataset` tiene tres niveles de dificultad:

| Dificultad | Qué la caracteriza |
|---|---|
| `easy` | Las oraciones de distintos autores tratan temas distintos, así que el tema ayuda. |
| `medium` | La variedad temática es pequeña: hay que fijarse más en el estilo. |
| `hard` | Todas las oraciones tratan el mismo tema: solo queda el estilo. |

Cada nivel tiene `train`, `validation` y `test`. Cada problema consta de:
- `problem-N.txt`: una oración por línea.
- `truth-problem-N.json`: `{"authors": k, "changes": [...]}`, con `len(changes) = nº oraciones − 1`.

> **Nota sobre la carga.** En el repositorio, los `.txt` de la carpeta `mawsa_full_dataset/` son *punteros de Git LFS* (contienen `version https://git-lfs...` en lugar del texto). Por eso los datos se leen **directamente del `mawsa_full_dataset.zip` original**. Además es mucho más rápido: un único fichero frente a ~94.000 ficheros pequeños en Windows. Si la carpeta tuviera el texto real (por ejemplo, tras `git lfs checkout`), el cargador la usaría como alternativa.

## 1. Entorno

In [ ]:
# Librerías
# En Windows, torch debe importarse ANTES que numpy/matplotlib/sklearn: si no, puede fallar la carga
# de sus DLLs (OSError WinError 1114 al cargar c10.dll)
import torch
import os
import gc
import re
import json
import math
import time
import random
import zipfile
from pathlib import Path

os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from tqdm.auto import tqdm

transformers.logging.set_verbosity_error()
pd.set_option("display.max_colwidth", 120)
print(f"torch {torch.__version__} | transformers {transformers.__version__} | pandas {pd.__version__}")

In [ ]:
# GPU: comprobamos qué soporta (igual que en el cuaderno de optimización)
assert torch.cuda.is_available(), "Este notebook necesita una GPU con CUDA"
device = torch.device("cuda")

GPU_NAME = torch.cuda.get_device_name(0)
CAPABILITY = torch.cuda.get_device_capability(0)
GPU_MEMORY_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
IS_AMPERE_OR_NEWER = CAPABILITY >= (8, 0)
# bfloat16 tiene el mismo rango que float32 -> no necesita GradScaler. En GPUs anteriores a Ampere: float16 + loss scaling
AMP_DTYPE = torch.bfloat16 if IS_AMPERE_OR_NEWER else torch.float16

# TF32: las multiplicaciones de matrices en float32 que queden fuera de autocast usan tensor cores
torch.set_float32_matmul_precision("high")

print(f"GPU: {GPU_NAME} | compute capability {CAPABILITY} | {GPU_MEMORY_GB:.1f} GB")
print(f"Precisión mixta: {AMP_DTYPE}")

### Configuración

Todos los hiperparámetros están en esta celda. Si se define la variable de entorno `SMOKE_TEST=1`, el notebook se ejecuta con muy pocos datos para comprobar en un par de minutos que el código funciona de principio a fin.

In [ ]:
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"

SEED = 42
ZIP_PATH = Path("mawsa_full_dataset.zip")
DATA_DIR = Path("mawsa_full_dataset/full_dataset")
DIFFICULTIES = ["easy", "medium", "hard"]
SPLITS = ["train", "validation", "test"]

CACHE_DIR = Path("cache_smoke" if SMOKE_TEST else "cache")
MODEL_DIR = Path("models_smoke/roberta-base-pairs" if SMOKE_TEST else "models/roberta-base-pairs")
RESULTS_DIR = Path("results_smoke" if SMOKE_TEST else "results")

# --- Datos ---
MODEL_NAME = "roberta-base"
MAX_LEN = 128                     # tokens del par completo (cubre ~95-98% de los pares sin truncar, ver EDA)
N_TRAIN_PAIRS_PER_DIFF = 40_000   # pares de train muestreados por dificultad (hay 0,55M-1,3M)
N_VAL_MONITOR_PER_DIFF = 5_000    # pares de validación para elegir el mejor checkpoint al final de cada época
EVAL_PROBLEMS_PER_DIFF = None     # None = validación y test completos

# --- Entrenamiento ---
EPOCHS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
MAX_GRAD_NORM = 1.0
EFFECTIVE_BATCH_SIZE = 32         # batch "lógico": batch físico x pasos de acumulación
EVAL_BATCH_SIZE = 128
USE_CLASS_WEIGHTS = True          # compensa el desbalance (en medium solo el 4% son cambios)
GRADIENT_CHECKPOINTING = False    # actívalo solo si el batch físico que cabe es muy pequeño
MEMORY_BUDGET_FRACTION = 0.85     # fracción de la VRAM que permitimos usar (en Windows, pasarse = memoria compartida lenta)
RETRAIN = False                   # False: si ya existe un modelo guardado, se carga en lugar de reentrenar
LOG_EVERY = 50                    # pasos de optimizador entre registros de la pérdida

if SMOKE_TEST:
    N_TRAIN_PAIRS_PER_DIFF, N_VAL_MONITOR_PER_DIFF, EVAL_PROBLEMS_PER_DIFF = 400, 200, 15
    EPOCHS, LOG_EVERY, RETRAIN = 1, 5, True

for d in (CACHE_DIR, MODEL_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("SMOKE_TEST =", SMOKE_TEST)

In [ ]:
# Utilidades: semillas, medición de tiempo/memoria y tablas de resultados


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def release(*names):
    # Borra variables globales por nombre y devuelve su memoria de GPU
    for name in names:
        globals().pop(name, None)
    gc.collect()
    torch.cuda.empty_cache()


class Measure:
    # Context manager que mide segundos de reloj y memoria pico de GPU (GB) de un bloque
    def __enter__(self):
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        self.start = time.perf_counter()
        return self

    def __exit__(self, *exc):
        torch.cuda.synchronize()
        self.seconds = time.perf_counter() - self.start
        self.peak_gb = torch.cuda.max_memory_allocated() / 1024**3


SCOREBOARD = []   # una fila por experimento: F1-macro por dificultad en test
COSTS = []        # una fila por medición de coste (tiempo, memoria, throughput)


def record_score(name, scores, train_info="", conclusion=""):
    SCOREBOARD[:] = [r for r in SCOREBOARD if r["experimento"] != name]  # re-ejecutar una celda reemplaza su fila
    SCOREBOARD.append({"experimento": name, "entrenamiento": train_info, **scores, "conclusión": conclusion})


def record_cost(name, **values):
    COSTS[:] = [r for r in COSTS if r["qué"] != name]
    COSTS.append({"qué": name, **values})


seed_everything()

## 2. Carga del dataset

Diseño del cargador:
- Lee del `.zip` en *streaming*: abre un problema, extrae sus pares y lo descarta. Con 16 GB de RAM no conviene tener en memoria los ~400 millones de caracteres del corpus.
- Primero lee solo las **etiquetas** (`truth-*.json`, pequeñas) de todos los problemas. Con ellas hace la EDA de etiquetas y el muestreo de train.
- Después lee el **texto** solo de los pares necesarios: la muestra de train y la validación y el test completos.
- Comprueba en cada problema que `nº oraciones − 1 == len(changes)`.
- Guarda en caché (`parquet`) los DataFrames de pares, de modo que la segunda ejecución tarda segundos.

In [ ]:
PROBLEM_RE = re.compile(r"(easy|medium|hard)/(train|validation|test)/problem-(\d+)\.txt$")
LFS_PREFIX = "version https://git-lfs"


class DatasetReader:
    # Lee problemas del zip original; si no existe, de la carpeta (siempre que no sean punteros LFS)

    def __init__(self, zip_path=ZIP_PATH, data_dir=DATA_DIR):
        self.zip = zipfile.ZipFile(zip_path) if zip_path.exists() else None
        self.data_dir = data_dir
        self.index = {}  # (difficulty, split, problem_id) -> ruta del .txt
        names = self.zip.namelist() if self.zip else [p.as_posix() for p in data_dir.rglob("problem-*.txt")]
        for name in names:
            m = PROBLEM_RE.search(name)
            if m:
                self.index[(m.group(1), m.group(2), int(m.group(3)))] = name
        print(f"Fuente: {'zip ' + str(zip_path) if self.zip else 'carpeta ' + str(data_dir)} | {len(self.index)} problemas")

    def _read(self, name):
        data = self.zip.read(name) if self.zip else Path(name).read_bytes()
        return data.decode("utf-8")

    def truth(self, key):
        return json.loads(self._read(self.index[key].replace("problem-", "truth-problem-").replace(".txt", ".json")))

    def sentences(self, key):
        text = self._read(self.index[key])
        if text.startswith(LFS_PREFIX):
            raise RuntimeError(f"{self.index[key]} es un puntero de Git LFS, no el texto. Usa el zip o 'git lfs checkout'.")
        return [s.rstrip("\r") for s in text.split("\n")]

    def keys(self, difficulty, split):
        return sorted(k for k in self.index if k[0] == difficulty and k[1] == split)


reader = DatasetReader()

In [ ]:
# 2.1 Etiquetas de todos los problemas (sin texto): base para la EDA de etiquetas y el muestreo
def load_truths(reader):
    path = CACHE_DIR / "truths.pkl"
    if path.exists():
        return pd.read_pickle(path)
    rows = []
    for difficulty in DIFFICULTIES:
        for split in SPLITS:
            for key in tqdm(reader.keys(difficulty, split), desc=f"{difficulty}/{split}", leave=False):
                truth = reader.truth(key)
                rows.append({"difficulty": difficulty, "split": split, "problem_id": key[2],
                             "authors": truth["authors"], "changes": truth["changes"],
                             "n_pairs": len(truth["changes"])})
    truths = pd.DataFrame(rows)
    truths.to_pickle(path)
    return truths


truths = load_truths(reader)
print(truths.shape)
truths.head()

In [ ]:
# 2.2 Construcción de pares (sent_a, sent_b, label) leyendo el texto en streaming
def build_pairs(reader, truths_subset, keep=None, desc=""):
    # keep: dict problem_id -> array de índices de pares a conservar (None = todos los pares)
    rows, mismatches = [], 0
    for r in tqdm(truths_subset.itertuples(index=False), total=len(truths_subset), desc=desc, leave=False):
        sentences = reader.sentences((r.difficulty, r.split, r.problem_id))
        if len(sentences) - 1 != len(r.changes):
            mismatches += 1
            continue
        indices = range(len(r.changes)) if keep is None else keep[r.problem_id]
        for i in indices:
            rows.append((r.difficulty, r.split, r.problem_id, int(i), r.authors,
                         sentences[i], sentences[i + 1], int(r.changes[i])))
    if mismatches:
        print(f"[aviso] {mismatches} problemas descartados por no cuadrar oraciones y etiquetas")
    return pd.DataFrame(rows, columns=["difficulty", "split", "problem_id", "idx", "authors",
                                       "sent_a", "sent_b", "label"])


def sample_train_keep(truths_train, n_pairs, seed):
    # Muestreo uniforme de n_pairs pares entre TODOS los pares de train de una dificultad
    rng = np.random.default_rng(seed)
    offsets = np.concatenate([[0], np.cumsum(truths_train.n_pairs.values)])
    chosen = np.sort(rng.choice(offsets[-1], size=min(n_pairs, offsets[-1]), replace=False))
    owner = np.searchsorted(offsets, chosen, side="right") - 1
    keep = {}
    for o, flat in zip(owner, chosen):
        keep.setdefault(truths_train.problem_id.values[o], []).append(flat - offsets[o])
    return truths_train[truths_train.problem_id.isin(keep)], keep


def cached(name, builder):
    path = CACHE_DIR / f"{name}.parquet"
    if path.exists():
        return pd.read_parquet(path)
    df = builder()
    df.to_parquet(path, index=False)
    return df


def load_split_pairs(split):
    parts = []
    for difficulty in DIFFICULTIES:
        t = truths[(truths.difficulty == difficulty) & (truths.split == split)]
        if split == "train":
            t, keep = sample_train_keep(t, N_TRAIN_PAIRS_PER_DIFF, SEED)
        else:
            keep = None
            if EVAL_PROBLEMS_PER_DIFF is not None:
                t = t.sample(n=min(EVAL_PROBLEMS_PER_DIFF, len(t)), random_state=SEED)
        parts.append(build_pairs(reader, t, keep, desc=f"{difficulty}/{split}"))
    return pd.concat(parts, ignore_index=True)


with Measure() as m:
    train_df = cached(f"train_pairs_{N_TRAIN_PAIRS_PER_DIFF}", lambda: load_split_pairs("train"))
    val_df = cached("validation_pairs", lambda: load_split_pairs("validation"))
    test_df = cached("test_pairs", lambda: load_split_pairs("test"))
print(f"Carga: {m.seconds:.0f} s | train {len(train_df):,} | validation {len(val_df):,} | test {len(test_df):,} pares")

In [ ]:
# 2.3 Comprobaciones de integridad
assert set(train_df.label.unique()) <= {0, 1}
assert not train_df[["sent_a", "sent_b"]].isna().any().any()
# Validación y test: cada problema cargado aporta exactamente sus n_pairs pares
for name, df in [("validation", val_df), ("test", test_df)]:
    got = df.groupby(["difficulty", "problem_id"]).size()
    expected = truths[truths.split == name].set_index(["difficulty", "problem_id"]).n_pairs.reindex(got.index)
    assert (got == expected).all(), name
print("Train por dificultad:", train_df.difficulty.value_counts().to_dict())
print("Pares vacíos (alguna oración en blanco):",
      int(((train_df.sent_a.str.strip() == "") | (train_df.sent_b.str.strip() == "")).sum()))

# Un ejemplo: los primeros pares de un problema de validación de 'hard'
example = val_df[(val_df.difficulty == "hard") & (val_df.problem_id == val_df.problem_id.iloc[-1])].head(6)
example[["idx", "authors", "sent_a", "sent_b", "label"]]

## 3. Análisis exploratorio

In [ ]:
# 3.1 Tamaño y desbalance por dificultad y partición (sobre TODAS las etiquetas, no solo la muestra)
truths["n_changes"] = truths.changes.apply(sum)
summary = (truths.groupby(["difficulty", "split"], sort=False)
           .agg(problemas=("problem_id", "count"), pares=("n_pairs", "sum"), cambios=("n_changes", "sum"),
                oraciones_por_doc=("n_pairs", lambda s: (s + 1).mean()), autores_medios=("authors", "mean")))
summary["% cambios"] = 100 * summary.cambios / summary.pares
summary.round(2)

In [ ]:
# 3.2 Distribuciones: autores por documento, oraciones por documento y % de cambios por dificultad
tr = truths[truths.split == "train"]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.countplot(data=tr, x="authors", hue="difficulty", hue_order=DIFFICULTIES, ax=axes[0])
axes[0].set_title("Autores por documento (train)")
sns.histplot(data=tr, x=tr.n_pairs + 1, hue="difficulty", hue_order=DIFFICULTIES, bins=60, element="step", ax=axes[1])
axes[1].set_title("Oraciones por documento (train)")
axes[1].set_xlabel("oraciones")
sums = tr.groupby("difficulty")[["n_changes", "n_pairs"]].sum().reindex(DIFFICULTIES)
rate = 100 * sums.n_changes / sums.n_pairs
rate.plot.bar(ax=axes[2], color=sns.color_palette()[:3])
axes[2].set_title("% de pares con cambio de autor (train)")
plt.tight_layout()
plt.show()

In [ ]:
# 3.3 Longitud en tokens de RoBERTa: justifica MAX_LEN
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
sample = train_df.groupby("difficulty").sample(n=min(5000, len(train_df) // 3), random_state=SEED)
len_a = np.array([len(x) for x in tokenizer(sample.sent_a.tolist(), add_special_tokens=False)["input_ids"]])
len_b = np.array([len(x) for x in tokenizer(sample.sent_b.tolist(), add_special_tokens=False)["input_ids"]])
sample = sample.assign(pair_tokens=len_a + len_b + 4)  # <s> A </s></s> B </s>

print(sample.groupby("difficulty").pair_tokens.describe(percentiles=[.5, .9, .95, .99]).round(1))
print(f"\n% de pares truncados con MAX_LEN={MAX_LEN}:",
      (100 * sample.groupby("difficulty").pair_tokens.apply(lambda s: (s > MAX_LEN).mean())).round(2).to_dict())
sns.histplot(data=sample, x="pair_tokens", hue="difficulty", bins=80, element="step")
plt.axvline(MAX_LEN, color="k", linestyle=":", label=f"MAX_LEN={MAX_LEN}")
plt.xlim(0, 300)
plt.legend()
plt.title("Tokens por par (muestra de train)")
plt.show()

**Observaciones del EDA** (se completan tras ejecutar):
- **Desbalance**: el porcentaje de cambios varía mucho entre dificultades: ~30 % en *easy*, ~4 % en *medium* y ~21 % en *hard*. Por eso se usa una pérdida ponderada por clase y se ajusta el umbral de decisión en validación.
- **Longitud**: una oración mide ~34 tokens de mediana. Con `MAX_LEN = 128` se truncan muy pocos pares, y la truncación `longest_first` recorta la oración más larga, conservando el principio de ambas.
- **Tamaño**: train tiene ~2,4 M de pares. Una época completa en una GPU de 4 GB llevaría muchas horas, por eso se **submuestrean** 40k pares por dificultad. La validación y el test se evalúan **completos**.

## 4. Preprocesado: tokenización y batches eficientes

- **Entrada tipo *cross-encoder***: `<s> oración_i </s></s> oración_i+1 </s>`. La atención compara directamente los tokens de las dos oraciones, sin pasar antes por un vector resumen de cada una.
- **No se normaliza el texto** (ni minúsculas ni limpieza de puntuación). Las comillas tipográficas, las mayúsculas, los guiones y la puntuación son precisamente señales de estilo, y el BPE a nivel de byte de RoBERTa las conserva.
- **Padding dinámico y batches agrupados por longitud**: cada batch se rellena solo hasta su secuencia más larga, y las secuencias de longitud parecida van juntas. Se procesan muchos menos tokens de *padding* que rellenando siempre a `MAX_LEN`.

In [ ]:
def encode(df, batch=20_000):
    ids = []
    for start in tqdm(range(0, len(df), batch), desc="Tokenizando", leave=False):
        chunk = df.iloc[start:start + batch]
        ids += tokenizer(chunk.sent_a.tolist(), chunk.sent_b.tolist(), truncation="longest_first",
                         max_length=MAX_LEN)["input_ids"]
    return {"input_ids": ids, "length": np.fromiter((len(x) for x in ids), dtype=np.int32, count=len(ids)),
            "labels": df.label.to_numpy(dtype=np.int64)}


def length_grouped_batches(lengths, batch_size, shuffle, seed=SEED, chunk_factor=50):
    # shuffle=True: baraja, ordena por longitud dentro de bloques grandes y baraja el orden de los batches.
    # shuffle=False: ordena todo por longitud (inferencia: mínimo padding; el orden original se restaura después).
    if not shuffle:
        order = np.argsort(lengths, kind="stable")
        return [order[i:i + batch_size] for i in range(0, len(order), batch_size)]
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(lengths))
    chunk = batch_size * chunk_factor
    batches = []
    for i in range(0, len(order), chunk):
        block = order[i:i + chunk]
        block = block[np.argsort(lengths[block], kind="stable")]
        batches += [block[j:j + batch_size] for j in range(0, len(block), batch_size)]
    rng.shuffle(batches)
    return batches


def pad_batch(input_ids, idx):
    seqs = [input_ids[i] for i in idx]
    width = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), width), tokenizer.pad_token_id, dtype=torch.long)
    mask = torch.zeros((len(seqs), width), dtype=torch.long)
    for row, s in enumerate(seqs):
        ids[row, :len(s)] = torch.tensor(s)
        mask[row, :len(s)] = 1
    return ids.pin_memory().to(device, non_blocking=True), mask.pin_memory().to(device, non_blocking=True)


# Subconjunto de validación para monitorizar cada época (estratificado por dificultad)
monitor_df = (val_df.groupby("difficulty")
              .sample(n=min(N_VAL_MONITOR_PER_DIFF, val_df.difficulty.value_counts().min()), random_state=SEED)
              .reset_index(drop=True))

with Measure() as m:
    train_enc, monitor_enc = encode(train_df), encode(monitor_df)
    val_enc, test_enc = encode(val_df), encode(test_df)
print(f"Tokenización: {m.seconds:.0f} s")

batches = length_grouped_batches(train_enc["length"], 32, shuffle=True)
real = sum(len(b) * train_enc["length"][b].max() for b in batches)
print(f"Tokens procesados por época: con padding a MAX_LEN {len(train_df) * MAX_LEN:,} | "
      f"con padding dinámico agrupado {real:,} ({100 * real / (len(train_df) * MAX_LEN):.0f}%)")

In [ ]:
# Distribución de clases en lo que realmente se usa
pd.concat({name: df.groupby("difficulty").label.agg(pares="size", cambios="sum", pct=lambda s: 100 * s.mean())
           for name, df in [("train (muestra)", train_df), ("validation", val_df), ("test", test_df)]},
          axis=1).reindex(DIFFICULTIES).round(2)

## 5. Métrica y baselines triviales

La métrica oficial de PAN es el **F1-macro** sobre los pares de cada dificultad, es decir, la media del F1 de la clase «cambio» y del F1 de la clase «sin cambio». Se implementa vectorizada con numpy para poder barrer umbrales rápidamente.

In [ ]:
def macro_f1(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    f1s = []
    for c in (0, 1):
        tp = np.sum((y_pred == c) & (y_true == c))
        fp = np.sum((y_pred == c) & (y_true != c))
        fn = np.sum((y_pred != c) & (y_true == c))
        f1s.append(2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 0.0)
    return float(np.mean(f1s))


def f1_by_difficulty(df, y_pred):
    y_pred = np.asarray(y_pred)
    scores = {f"F1 {d}": macro_f1(df.label.values[(df.difficulty == d).values], y_pred[(df.difficulty == d).values])
              for d in DIFFICULTIES}
    scores["F1 medio"] = float(np.mean(list(scores.values())))
    return scores


# Baseline 0: nunca hay cambio
record_score("Baseline: siempre «sin cambio»", f1_by_difficulty(test_df, np.zeros(len(test_df), dtype=int)), "—",
             "Cota inferior trivial")

# Baseline 1: aleatorio con la tasa de cambios de train de cada dificultad
rng = np.random.default_rng(SEED)
train_sums = truths[truths.split == "train"].groupby("difficulty")[["n_changes", "n_pairs"]].sum()
train_rate = train_sums.n_changes / train_sums.n_pairs
random_pred = (rng.random(len(test_df)) < test_df.difficulty.map(train_rate).values).astype(int)
record_score("Baseline: aleatorio estratificado", f1_by_difficulty(test_df, random_pred),
             "tasa de cambios de train", "Azar informado: F1 ≈ 0,5")
pd.DataFrame(SCOREBOARD).round(4)

## 6. Fine-tuning de RoBERTa-base como *cross-encoder* de pares

### Justificación del modelo
- **RoBERTa-base** (125 M de parámetros) es un *encoder* bidireccional preentrenado con más datos y durante más tiempo que BERT. En las ediciones anteriores de esta tarea en PAN, los participantes con mejores resultados usaron mayoritariamente *encoders* tipo RoBERTa/DeBERTa ajustados sobre pares de oraciones.
- **Tokenizador BPE a nivel de byte**: no tiene tokens `[UNK]` y conserva exactamente la puntuación, las comillas tipográficas, las mayúsculas y los espacios, que son señales estilométricas.
- **Cross-encoder**: el par se codifica conjuntamente, de modo que la atención cruzada compara directamente léxico, registro y puntuación de ambas oraciones. La representación de `<s>` alimenta un clasificador lineal de 2 clases.

### Optimización para la GPU (RTX 3050, 4 GB), según el cuaderno de optimización

| Técnica | Qué aporta aquí |
|---|---|
| **AMP en bfloat16** (`torch.autocast`) | Las multiplicaciones de matrices van en 16 bits sobre tensor cores: más velocidad y menos memoria de activaciones. Los pesos maestros siguen en float32. bf16 tiene el rango de float32, así que no necesita `GradScaler` (sí lo necesitaría en una GPU pre-Ampere con fp16). |
| **TF32** | Las operaciones float32 que quedan fuera de autocast también usan tensor cores. |
| **Atención SDPA** (`attn_implementation="sdpa"`) | Kernels fusionados de PyTorch (*memory-efficient* / *flash*): la atención no materializa la matriz N×N completa. |
| **AdamW fusionado** (`fused=True`) | Un solo kernel para actualizar todos los parámetros: menos lanzamientos de kernel por paso. |
| **Padding dinámico + agrupación por longitud** | Se procesan muchos menos tokens de relleno (ver sección 4). |
| **Batch físico elegido midiendo memoria + acumulación de gradiente** | Se busca el mayor batch que cabe con margen. En Windows, pasarse de la VRAM no da error: el driver usa memoria compartida y el entrenamiento se vuelve muchísimo más lento. Se acumulan gradientes hasta el batch efectivo de 32. |
| **Gradient checkpointing** (opcional) | Recalcula activaciones en el *backward*: cambia tiempo por memoria. Aquí no hace falta (ver la tabla de la frontera de batch). |

**¿Por qué no QLoRA?** La cuantización a 4 bits y LoRA están pensadas para modelos de miles de millones de parámetros. RoBERTa-base ocupa ~0,5 GB en float32, y el fine-tuning completo (pesos, gradientes y los dos momentos de Adam, ~2 GB) cabe en 4 GB. Además, en *encoders* pequeños el fine-tuning completo suele dar mejor resultado que los adaptadores. LoRA queda como posible experimento posterior.

In [ ]:
def build_model(name_or_path=MODEL_NAME, gradient_checkpointing=GRADIENT_CHECKPOINTING):
    try:
        model = AutoModelForSequenceClassification.from_pretrained(name_or_path, num_labels=2, attn_implementation="sdpa")
    except (ValueError, ImportError):
        model = AutoModelForSequenceClassification.from_pretrained(name_or_path, num_labels=2)
    if gradient_checkpointing:
        model.gradient_checkpointing_enable()
    return model.to(device)


def make_optimizer(model, lr=LEARNING_RATE):
    # Sin weight decay en sesgos y LayerNorm (parámetros 1-D)
    decay = [p for p in model.parameters() if p.requires_grad and p.ndim >= 2]
    no_decay = [p for p in model.parameters() if p.requires_grad and p.ndim < 2]
    return torch.optim.AdamW([{"params": decay, "weight_decay": WEIGHT_DECAY},
                              {"params": no_decay, "weight_decay": 0.0}], lr=lr, fused=True)


probe = build_model()
n_params = sum(p.numel() for p in probe.parameters())
print(f"{MODEL_NAME}: {n_params / 1e6:.1f} M parámetros | atención: {probe.config._attn_implementation} | "
      f"pesos fp32: {probe.get_memory_footprint() / 1024**3:.2f} GB")
release("probe")

### Frontera del tamaño de batch

Se prueba cada tamaño de batch con el peor caso (todas las secuencias de longitud `MAX_LEN`) durante dos pasos de entrenamiento completos. El segundo paso es el que reserva los estados de Adam. Se elige el mayor batch cuya memoria reservada quede por debajo del `MEMORY_BUDGET_FRACTION` de la VRAM.

In [ ]:
def try_batch(batch_size, seq_len=MAX_LEN):
    model = optimizer = None
    try:
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        model = build_model()
        model.train()
        optimizer = make_optimizer(model)
        ids = torch.randint(5, tokenizer.vocab_size - 1, (batch_size, seq_len), device=device)
        mask = torch.ones_like(ids)
        labels = torch.randint(0, 2, (batch_size,), device=device)
        start = time.perf_counter()
        for _ in range(2):
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                loss = model(input_ids=ids, attention_mask=mask, labels=labels).loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
        torch.cuda.synchronize()
        seconds = (time.perf_counter() - start) / 2
        peak = torch.cuda.max_memory_reserved() / 1024**3
        return {"batch": batch_size, "cabe": peak < MEMORY_BUDGET_FRACTION * GPU_MEMORY_GB,
                "memoria reservada pico (GB)": round(peak, 2), "s/paso (peor caso)": round(seconds, 3)}
    except torch.OutOfMemoryError:
        return {"batch": batch_size, "cabe": False, "memoria reservada pico (GB)": float("nan"), "s/paso (peor caso)": float("nan")}
    finally:
        del model, optimizer
        gc.collect()
        torch.cuda.empty_cache()


frontier = []
for bs in [b for b in (64, 32, 16, 8, 4) if b <= EFFECTIVE_BATCH_SIZE]:
    frontier.append(try_batch(bs))
    if frontier[-1]["cabe"]:
        break
frontier = pd.DataFrame(frontier)
fitting = frontier[frontier.cabe]
PHYSICAL_BATCH_SIZE = int(fitting.batch.max()) if len(fitting) else 4
GRAD_ACCUM_STEPS = max(1, EFFECTIVE_BATCH_SIZE // PHYSICAL_BATCH_SIZE)
print(f"Batch físico = {PHYSICAL_BATCH_SIZE} | acumulación = {GRAD_ACCUM_STEPS} | "
      f"batch efectivo = {PHYSICAL_BATCH_SIZE * GRAD_ACCUM_STEPS}")
frontier

In [ ]:
@torch.inference_mode()
def predict_proba(model, enc, batch_size=EVAL_BATCH_SIZE, desc="Inferencia"):
    # Probabilidad de la clase 1 (cambio de autor) para cada par, en el orden original
    model.eval()
    probs = np.empty(len(enc["length"]), dtype=np.float32)
    for idx in tqdm(length_grouped_batches(enc["length"], batch_size, shuffle=False), desc=desc, leave=False):
        ids, mask = pad_batch(enc["input_ids"], idx)
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            logits = model(input_ids=ids, attention_mask=mask).logits
        probs[idx] = torch.softmax(logits.float(), dim=-1)[:, 1].cpu().numpy()
    return probs


def class_weights_from(labels):
    # Pesos inversamente proporcionales a la frecuencia (normalizados a media 1)
    counts = np.bincount(labels, minlength=2).astype(np.float64)
    w = counts.sum() / (2 * counts)
    return torch.tensor(w, dtype=torch.float32, device=device)


def train_model(model, train_enc, monitor_df, monitor_enc, epochs=EPOCHS):
    seed_everything()
    optimizer = make_optimizer(model)
    n_batches = math.ceil(len(train_enc["length"]) / PHYSICAL_BATCH_SIZE)
    total_updates = epochs * math.ceil(n_batches / GRAD_ACCUM_STEPS)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total_updates), total_updates)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_DTYPE == torch.float16)
    weights = class_weights_from(train_enc["labels"]) if USE_CLASS_WEIGHTS else None
    loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
    labels_all = torch.from_numpy(train_enc["labels"])
    print("Pesos de clase:", None if weights is None else weights.tolist())

    history = {"update": [], "loss": [], "lr": [], "epoch": []}
    best_f1, update, running, n_running = -1.0, 0, torch.zeros((), device=device), 0
    for epoch in range(epochs):
        model.train()
        batches = length_grouped_batches(train_enc["length"], PHYSICAL_BATCH_SIZE, shuffle=True, seed=SEED + epoch)
        optimizer.zero_grad(set_to_none=True)
        bar = tqdm(batches, desc=f"Época {epoch + 1}/{epochs}")
        for i, idx in enumerate(bar):
            ids, mask = pad_batch(train_enc["input_ids"], idx)
            labels = labels_all[idx].to(device, non_blocking=True)
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                logits = model(input_ids=ids, attention_mask=mask).logits
            loss = loss_fn(logits.float(), labels) / GRAD_ACCUM_STEPS
            scaler.scale(loss).backward()
            running += loss.detach() * GRAD_ACCUM_STEPS
            n_running += 1
            if (i + 1) % GRAD_ACCUM_STEPS == 0 or i + 1 == len(batches):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                update += 1
                if update % LOG_EVERY == 0:  # .item() sincroniza CPU-GPU: solo cada LOG_EVERY pasos
                    avg = (running / n_running).item()
                    history["update"].append(update); history["loss"].append(avg)
                    history["lr"].append(scheduler.get_last_lr()[0]); history["epoch"].append(epoch + 1)
                    bar.set_postfix(loss=f"{avg:.4f}")
                    running.zero_(); n_running = 0

        probs = predict_proba(model, monitor_enc, desc="Monitor validación")
        scores = f1_by_difficulty(monitor_df, (probs >= 0.5).astype(int))
        history.setdefault("monitor", []).append({"epoch": epoch + 1, **scores})
        print(f"Época {epoch + 1}: " + " | ".join(f"{k} {v:.4f}" for k, v in scores.items()))
        if scores["F1 medio"] > best_f1:
            best_f1 = scores["F1 medio"]
            model.save_pretrained(MODEL_DIR)
            tokenizer.save_pretrained(MODEL_DIR)
            print(f"  -> mejor checkpoint guardado en {MODEL_DIR}")
    return history

In [ ]:
# Entrenamiento (o carga del checkpoint si ya existe y RETRAIN=False)
HISTORY_PATH = MODEL_DIR / "history.json"
if RETRAIN or not (MODEL_DIR / "config.json").exists():
    model = build_model()
    with Measure() as m:
        history = train_model(model, train_enc, monitor_df, monitor_enc)
    history["train_seconds"], history["train_peak_gb"] = m.seconds, m.peak_gb
    history["n_updates"] = history["update"][-1] if history["update"] else None
    HISTORY_PATH.write_text(json.dumps(history, indent=1))
    release("model")
else:
    history = json.loads(HISTORY_PATH.read_text())
    print("Modelo ya entrenado: se carga de disco (pon RETRAIN=True para reentrenar)")

record_cost("Fine-tuning RoBERTa-base", **{
    "segundos": round(history["train_seconds"]), "memoria pico (GB)": round(history["train_peak_gb"], 2),
    "pares de train": len(train_df), "épocas": EPOCHS, "batch físico": PHYSICAL_BATCH_SIZE,
    "pares/s": round(EPOCHS * len(train_df) / history["train_seconds"], 1)})
print(f"Entrenamiento: {history['train_seconds'] / 60:.1f} min | memoria pico {history['train_peak_gb']:.2f} GB")

In [ ]:
# Curvas de entrenamiento
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(history["update"], history["loss"])
axes[0].set_xlabel("paso de optimizador")
axes[0].set_ylabel("pérdida (media móvil)")
axes[0].set_title("Pérdida de entrenamiento")
mon = pd.DataFrame(history["monitor"]).set_index("epoch")
mon.plot(marker="o", ax=axes[1])
axes[1].set_title("F1-macro en el subconjunto de validación (umbral 0,5)")
axes[1].set_xticks(mon.index)
plt.tight_layout()
plt.show()
mon.round(4)

### Inferencia sobre validación y test completos

Se carga el **mejor checkpoint**, es decir, el de mayor F1 medio en el subconjunto de monitorización, y se calculan las probabilidades de todos los pares.

In [ ]:
model = build_model(MODEL_DIR)
with Measure() as m:
    val_probs = predict_proba(model, val_enc, desc="Validación")
record_cost("Inferencia validación", **{"segundos": round(m.seconds), "memoria pico (GB)": round(m.peak_gb, 2),
                                         "pares/s": round(len(val_df) / m.seconds, 1)})
test_probs = predict_proba(model, test_enc, desc="Test")
np.save(RESULTS_DIR / "roberta_val_probs.npy", val_probs)
np.save(RESULTS_DIR / "roberta_test_probs.npy", test_probs)
print(f"Inferencia validación: {m.seconds:.0f} s ({len(val_df) / m.seconds:.0f} pares/s)")

### Ajuste del umbral de decisión en validación

Con clases desbalanceadas, el umbral 0,5 no tiene por qué maximizar el F1-macro. El umbral se elige **solo con validación** y se aplica después a test, de dos maneras:
- **Global**: un único umbral que maximiza el F1 medio de las tres dificultades.
- **Por dificultad**: un umbral para cada nivel. Es legítimo porque en PAN cada dificultad es un conjunto de datos separado y conocido.

In [ ]:
thresholds = np.round(np.arange(0.05, 0.96, 0.01), 2)
sweep = pd.DataFrame([{"umbral": t, **f1_by_difficulty(val_df, (val_probs >= t).astype(int))} for t in thresholds])
best_global = float(sweep.loc[sweep["F1 medio"].idxmax(), "umbral"])
best_per_diff = {d: float(sweep.loc[sweep[f"F1 {d}"].idxmax(), "umbral"]) for d in DIFFICULTIES}
print("Umbral global:", best_global, "| por dificultad:", best_per_diff)

sweep.set_index("umbral").plot(figsize=(8, 4))
plt.axvline(0.5, color="grey", linestyle=":")
plt.axvline(best_global, color="k", linestyle="--", label=f"global = {best_global}")
plt.ylabel("F1-macro (validación)")
plt.legend()
plt.show()

In [ ]:
# Evaluación final en TEST
train_info = f"{N_TRAIN_PAIRS_PER_DIFF:,} pares/dificultad, {EPOCHS} épocas"
per_diff_thr = test_df.difficulty.map(best_per_diff).values
variants = {
    "RoBERTa-base pares (umbral 0,5)": test_probs >= 0.5,
    f"RoBERTa-base pares (umbral global {best_global})": test_probs >= best_global,
    "RoBERTa-base pares (umbral por dificultad)": test_probs >= per_diff_thr,
}
for name, pred in variants.items():
    record_score(name, f1_by_difficulty(test_df, pred.astype(int)), train_info)

test_pred = (test_probs >= per_diff_thr).astype(int)
pd.DataFrame(SCOREBOARD).round(4)

In [ ]:
# Informe por clase y matrices de confusión (variante con umbral por dificultad)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, d in zip(axes, DIFFICULTIES):
    mask = (test_df.difficulty == d).values
    print(f"===== {d} (umbral {best_per_diff[d]}) =====")
    print(classification_report(test_df.label.values[mask], test_pred[mask], target_names=["sin cambio", "cambio"], digits=4))
    ConfusionMatrixDisplay(confusion_matrix(test_df.label.values[mask], test_pred[mask], normalize="true"),
                           display_labels=["sin cambio", "cambio"]).plot(ax=ax, colorbar=False, values_format=".2f")
    ax.set_title(d)
plt.tight_layout()
plt.show()

In [ ]:
# Análisis de errores: F1-macro en test según el número de autores del documento
by_authors = (test_df.assign(pred=test_pred)
              .groupby(["difficulty", "authors"])
              .apply(lambda g: pd.Series({"pares": len(g), "% cambios": 100 * g.label.mean(),
                                          "F1-macro": macro_f1(g.label, g.pred)}), include_groups=False)
              .round(3))
by_authors.unstack(0)["F1-macro"]

In [ ]:
# Ejemplos de errores en 'hard' (falsos positivos con mayor probabilidad y falsos negativos con menor)
errors = test_df.assign(prob=test_probs, pred=test_pred)
errors = errors[(errors.difficulty == "hard") & (errors.label != errors.pred)]
pd.concat([errors[errors.label == 0].nlargest(4, "prob"), errors[errors.label == 1].nsmallest(4, "prob")])[
    ["label", "prob", "sent_a", "sent_b"]]

## 7. Costes computacionales y comparación con el estado del arte

In [ ]:
pd.DataFrame(COSTS)

**Comparación con resultados publicados** (se completa en el paso de evaluación): la tabla de resultados del *overview* de PAN de Multi-Author Writing Style Analysis (ediciones 2025/2026) y de los *notebooks* de los participantes, junto a los F1 por dificultad obtenidos aquí.

In [ ]:
# Tabla resumen lista para copiar en la primera celda del notebook
board = pd.DataFrame(SCOREBOARD)
cols = ["F1 easy", "F1 medium", "F1 hard", "F1 medio"]
lines = ["| # | Experimento | Entrenamiento | " + " | ".join(cols) + " | Conclusión |",
         "|---|---|---|" + "---|" * len(cols) + "---|"]
for i, r in board.iterrows():
    lines.append(f"| {i} | {r['experimento']} | {r['entrenamiento']} | "
                 + " | ".join(f"{r[c]:.4f}" for c in cols) + f" | {r['conclusión'] or '_completar_'} |")
print("\n".join(lines))
board.to_csv(RESULTS_DIR / "scoreboard.csv", index=False)

## 8. Conclusiones

_(Se completan tras ejecutar los experimentos.)_